# Data cleaning and preparations

### TODO
1. [ ] Decide what to do with missing values (e.g. january 1-16 values in 1_tahe)
2. [x] Decide whether to incorporate 2026 as well. Where and how.
3. [x] Decide whether to incorporate recently added signs from after 2023.
4. [x] Rename filename column to sign label or something obvious.
5. [ ] Decide if and how to organize getting data for 2021-2024.

## Installation
* install python duckdb for anaconda (version 1.5.6 specifically, not the newest one)
* create a motherduck account and join the organization from link sent to UT email
* generate a motherduck read-write token for yourself under the sandra_md org and add it to anaconda env variables

In [201]:
import glob
import pandas as pd
from pathlib import Path
import duckdb
from datetime import timedelta
from functools import reduce

## Creating a database
The following code was used to generate a DuckDB database from the data sent to us by the Tartu city government (the data is also available online: https://avaandmed-tartu.hub.arcgis.com/datasets/Tartu::uus-kiirustablood-kuvamine/explore) and upload it to MotherDuck for use without downloading. This was done due to the large volume of data (76 million rows) and API (1000 rows/request) and CSV download (10 000 rows/download) limits. These limits were the reason we requested the data separately instead of downloading it ourselves. NB! The council only sent us data from 2025–2026 (34 million rows total). This code was created with help from AI (Claude).

```
print(duckdb.__version__) # check it's 1.5.6
```

```
folder = Path("data/Kiirustablood_arhiiv_2")

for f in folder.rglob("*.csv"): # rglob also searches in nested folders
    df = pd.read_csv(f, sep=";")
    df["filename"] = f.stem        # copy the file's name to a new column for each row, no .csv extension
    # this is because the file name is the label (tunnus) of the speed sign, which we need (the file itself doesn't contain it)
    df.to_csv(f, sep=";", index=False)  
```

```
con = duckdb.connect("kiirustablood_2526_new.duckdb") # connecting to local DuckDB

con.execute("""
    CREATE OR REPLACE TABLE all_data AS
    SELECT *
    FROM read_csv(
        'data/Kiirustablood_arhiiv_2/**/*.csv',
        delim = ';',
        header = true,
        union_by_name = true
    )
""")

print(con.execute("SELECT count(*) FROM all_data").fetchone()) # row count check
```

```
con = duckdb.connect("md:") # connects to MotherDuck remotely automatically using MOTHERDUCK_TOKEN from env variables or motherduck_token from anaconda
con.execute("CREATE DATABASE IF NOT EXISTS idsprojectdb")
con.execute("ATTACH 'kiirustablood_2526_new.duckdb' AS local_db (READ_ONLY)")

con.execute("""
    CREATE OR REPLACE TABLE idsprojectdb.main.all_data AS
    SELECT * FROM local_db.main.all_data
""")

print(con.execute("SELECT count(*) FROM idsprojectdb.main.all_data").fetchone()) # row count check (see if it matches local db row count)
```

```
con.execute("CREATE SHARE idsproject_share FROM idsprojectdb (ACCESS ORGANIZATION)")
```

```
# TEST to see if db works from online

con = duckdb.connect("md:")

df = con.execute("""
    SELECT *
    FROM idsprojectdb.main.all_data
    LIMIT 10
""").df()

print(df)
```

## Sharing a table you have created or attaching one by another person
### Use this to see all schemas shared with you   
*NB! Shares created by others are read-only. You need to create your own db with your work (you can use the contents of another person's db to create the data, but if you want to make and save changes, that's when the new db comes in).*
```python
con = duckdb.connect("md:")
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    print(con.execute("SELECT * FROM md_information_schema.shared_with_me").df())
con.execute("ATTACH 'kopeeri_siia_pikk_sharei_kood_teisena_väljastatud_lõigust'")
```

### Use this to share   
```python
con.execute("""
    CREATE OR REPLACE SHARE sharei_nimi_mille_ise_praegu_valid FROM sinu_db_täpne_nimi_kuhu_tabeli_tegid
    (ACCESS ORGANIZATION, VISIBILITY DISCOVERABLE, UPDATE AUTOMATIC)
""")
print(con.execute("LIST SHARES").fetchall())
```
-> check that it appears  

Then on the MotherDuck website, find your share, click on it and click "attach",
then three dots -> "copy share link" -> paste it as a comment in Jupyter or share with others in some other way.

## Adding weather data
The following code was used to add hourly weather data to the hourly_db_share and create hourly_with_weather. In other words, we took weather data from Keskkonnaagentuur (https://www.ilmateenistus.ee/kliima/ajaloolised-ilmaandmed/) by hour, adjusted timestamps to the best of our knowledge and abilities (original traffic data from our main dataset lacks documentation, particulary about what timezone the timestamp is) and added everything to hourly traffic data. There are also some informative columns added in the process, like if any dates were adjusted for time shifts (EET vs. EEST) – these should be removed when using a predictive model.

```python
#con = duckdb.connect("md:")
#print(con.execute("SELECT count(*) FROM hourly_db_share.hourly").fetchone()) # working with 153k rows
```

```python
# decided temperature (winter = snow + ice), percipitation(rain/snow), sunshine (dark winters, nights), wind (snow storm, fallen trees, thunderstorm)
# probably affect driving speed most, the data source has more possible attributes,
# e.g. air pressure, that don't seem as significant

# before this I already attached the hourly db from motherduck
# nb! separator is semicolon
# nb! we have float values with comma instead of period
air_temperature_df = pd.read_csv('data/air-temperature-2526.csv',sep=';')
# value measured once at full hour, unit is celsius
percipitation_df = pd.read_csv('data/percipitation-2526.csv',sep=';')
# hourly sum, unit is mm
sunshine_duration_df = pd.read_csv('data/sunshine-duration-2526.csv',sep=';')
# hourly sum, unit is minutes
wind_speed_df = pd.read_csv('data/wind-speed-2526.csv',sep=';')
# last 10 min avg, unit is m/s

weather_dfs = [air_temperature_df, percipitation_df, sunshine_duration_df, wind_speed_df]

air_temperature_df.shape # 16 560 so we got all the rows successfully, no 10k limit here!
```

We have some duplicate rows in the weather data. The code below checks for duplicate values. We can see that they don't conflict except one that differs only slightly. We take the maximum here as it's likely a correction.

```python
def helper(df):
    cols = ['jaam_nimi - Station name, local (est)','element_nimi_eng - Element name (eng)','element_yhik_eng - Unit of measurement (eng)']
    df2 = df.drop(columns=cols)
    df2.columns = ['year', 'month', 'day', 'hour', 'value']
    df2['time_utc'] = (pd.to_datetime(df2[['year', 'month', 'day']]) 
                      + pd.to_timedelta(df2['hour'], unit='h')).dt.tz_localize('UTC')
    return df2
```

```python
for df in weather_dfs:
    df2 = helper(df)
    d = df2[df2['time_utc'].duplicated(keep=False)].sort_values('time_utc')
    # does each duplicated timestamp carry more than one distinct value?
    conflicts = d.groupby('time_utc')['value'].nunique(dropna=False)
    print('duplicated rows:', len(d),
          '| timestamps with conflicting values:', (conflicts > 1).sum())
    print(d[d['time_utc'].isin(conflicts[conflicts > 1].index)].head(10))
    print()
```

**Output**   

duplicated rows: 96 | timestamps with conflicting values: 0
Empty DataFrame
Columns: [year, month, day, hour, value, time_utc]
Index: []

duplicated rows: 96 | timestamps with conflicting values: 1
      year  month  day  hour value                  time_utc
6133  2026      3   11    10     0 2026-03-11 10:00:00+00:00
6132  2026      3   11    10   0,1 2026-03-11 10:00:00+00:00

duplicated rows: 96 | timestamps with conflicting values: 0
Empty DataFrame
Columns: [year, month, day, hour, value, time_utc]
Index: []

duplicated rows: 96 | timestamps with conflicting values: 0
Empty DataFrame
Columns: [year, month, day, hour, value, time_utc]
Index: []

The code bellow counts NaN values. Some values are missing, which we need to keep in mind and fill, but the gaps aren't large so we can interpolate.

```python
# print(wind_speed_df[wind_speed_df.isna().any(axis=1)]) # try this to see whole rows

print(wind_speed_df.isna().sum())
print(sunshine_duration_df.isna().sum())
print(percipitation_df.isna().sum())
print(air_temperature_df.isna().sum())
```

```python
# remove station name (always the same), element name (indicated in df name, later added as column name), unit (noted, later added to column name)

def reformat_weather_data(df, value_column_name='value'):
    cols = ['jaam_nimi - Station name, local (est)','element_nimi_eng - Element name (eng)','element_yhik_eng - Unit of measurement (eng)']
    df = df.drop(columns=cols)
    df.columns = ['year', 'month', 'day', 'hour', 'value']
    # add utc time for percision and tallinn time for joining to hourly db
    df['time_utc'] = (pd.to_datetime(df[['year', 'month', 'day']]) 
                      + pd.to_timedelta(df['hour'], unit='h')).dt.tz_localize('UTC')
    df['time_tallinn'] = df.apply(format_tallinn_timestamp, axis=1)
    # make str 4,4 into Float 4.4 (allows NaN)
    df["value"] = pd.to_numeric(df["value"].astype("string").str.replace(",", ".", regex=False), errors="coerce")
    
    # fill NaN values with closest values' averages
    s = df.groupby('time_utc')['value'].max().asfreq('h')
    filled = s.interpolate(method='time', limit_area='inside')
    out = pd.DataFrame({
        value_column_name: filled
    })
    out['time_tallinn'] = (out.index.tz_convert('Europe/Tallinn').tz_localize(None).astype('datetime64[us]'))
    return out.reset_index() # returns to standard indexing and makes time_utc a column again

air_temperature_df = reformat_weather_data(air_temperature_df, 'air_temperature_at_full_hour_celsius')
percipitation_df = reformat_weather_data(percipitation_df, 'percipitation_sum_mm')
sunshine_duration_df = reformat_weather_data(sunshine_duration_df, 'sunshine_duration_sum_min')
wind_speed_df = reformat_weather_data(wind_speed_df, 'wind_speed_last_10_min_avg_m_s')

print(air_temperature_df.dtypes)
print(air_temperature_df.head(10))
```

```python
# merge all weather df-s into one
frames = [air_temperature_df, wind_speed_df, percipitation_df, sunshine_duration_df]
# keep time_tallinn only in the first frame to avoid duplication errors
frames = [frames[0]] + [f.drop(columns='time_tallinn') for f in frames[1:]]
weather = reduce(lambda l, r: l.merge(r, on='time_utc', how='outer'), frames)
print(weather['time_utc'].is_unique, weather.shape)
print(weather.columns.tolist())
```

```python
weather_local = weather.sort_values('time_utc').copy()
weather_local['utc_hours_in_label'] = (
    weather_local.groupby('time_tallinn')['time_utc'].transform('size'))
weather_local = weather_local.drop_duplicates('time_tallinn', keep='first')
weather_local['time_tallinn'] = weather_local['time_tallinn'].astype('datetime64[us]')

print(weather_local['time_tallinn'].is_unique)
print(weather_local.loc[weather_local['utc_hours_in_label'] > 1, ['time_tallinn', 'time_utc']])
# expect exactly 2 rows: 2025-10-26 03:00 and 2026-10-25 03:00

con.sql("CREATE DATABASE IF NOT EXISTS hourly_with_weather")

con.register('weather_local', weather_local)

con.sql("""
CREATE OR REPLACE TABLE hourly_with_weather.main.all_data AS
WITH t AS (
    SELECT
        hdb.*,
        CAST(hdb.hour_start AS TIMESTAMP)                    AS ts,
        date_trunc('hour', CAST(hdb.hour_start AS TIMESTAMP)) AS ts_hour,
        CAST(hdb.hour_start AS DATE)                         AS d,
        hour(CAST(hdb.hour_start AS TIMESTAMP))              AS h
    FROM hourly_db_share.hourly AS hdb
),
keyed AS (
    SELECT
        t.*,
        CASE
            -- spring: 03:xx does not exist, the clock jumped 03:00 -> 04:00, so use the 04:00 weather
            WHEN d IN (DATE '2025-03-30', DATE '2026-03-29') AND h = 3
                THEN CAST(ts_hour + INTERVAL 1 HOUR AS TIMESTAMP)
            ELSE CAST(ts_hour AS TIMESTAMP)
        END AS join_hour,
        CASE
            WHEN d IN (DATE '2025-03-30', DATE '2026-03-29') AND h = 3
                THEN 'spring_gap_shifted_to_04'
            WHEN d IN (DATE '2025-10-26', DATE '2026-10-25') AND h = 3
                THEN 'autumn_repeated_hour'
            ELSE NULL
        END AS dst_event
    FROM t
)
SELECT
    k.* EXCLUDE (ts, ts_hour, d, h),
    w.* EXCLUDE (time_tallinn)          -- includes time_utc as the backup column
FROM keyed AS k
LEFT JOIN weather_local AS w
       ON CAST(w.time_tallinn AS TIMESTAMP) = k.join_hour
""")
```

```python
# AI recommended checks - all passed
con.sql("""
SELECT
    (SELECT count(*) FROM hourly_db_share.hourly)         AS traffic_rows,
    count(*)                                              AS merged_rows,      -- must be equal
    count(*) FILTER (WHERE air_temperature_at_full_hour_celsius IS NULL)              AS no_weather,
    count(*) FILTER (WHERE dst_event IS NOT NULL)         AS dst_rows
FROM hourly_with_weather.main.all_data
""").show()

con.sql("""
SELECT hour_start, join_hour, time_utc, utc_hours_in_label, dst_event
FROM hourly_with_weather.main.all_data
WHERE dst_event IS NOT NULL OR air_temperature_at_full_hour_celsius IS NULL
ORDER BY hour_start
""").show()

con.sql(
    """
    SELECT count(*) AS mismatches
    FROM hourly_with_weather.main.all_data
    WHERE time_utc IS NOT NULL
    AND (time_utc AT TIME ZONE 'Europe/Tallinn') <> join_hour;
    """
)
```

```python
con.execute("""
    CREATE OR REPLACE SHARE hourly_with_weather_share FROM hourly_with_weather
    (ACCESS ORGANIZATION, VISIBILITY DISCOVERABLE, UPDATE AUTOMATIC)
""")
print(con.execute("LIST SHARES").fetchall())
```

# ---------- OLD -------------

In [54]:
# function for turning TV_kiirustabloo.csv into a dataframe and removing unnecessary columns
def process_sign_info_file(filename: str) -> pd.DataFrame:
    df = pd.read_csv(filename)
    df_sub = df[['X','Y','Tunnus','Maks_kiirus']] # kas tahame koordinaate alles hoida?
    return df_sub

In [58]:
# 4 km/h is always subtracted from any speeding incidents (mõõtemääramatus)
# speed cameras register speedings starting from 7 km/h over the limit, mobile police radars from 10 km/h over
# currently default speeding tolerance is therefore 6 km/h, since it's the lowest you can get away with
# currently instances w/ speeds under 5 km/h and over 150 km/h are removed, we should assess if these bounds are suitable
# currently hours where less than 5 cars passed are removed, should assess if we should keep that
# weekday -> monday = 0
def process_measured_speeds_file(measured_speeds_file: str, sign_info_file="data/vana/TV_kiirustabloo.csv", min_passing_cars=5, 
                                 speeding_tolerance_kmh=6, min_speed=5, max_speed=150) -> pd.DataFrame:
    df = pd.read_csv(
        measured_speeds_file,
        usecols=["Tunnus", "V1", "Year", "Month", "Day", "Hour"],
        dtype={"V1": "float32", "Tunnus": "category"},
    )

    # eemalda mõõtmisvead(?) ja äärmused
    df = df[df["V1"].between(min_speed, max_speed)]

    # lisa piirang ja märgi ületajate arv
    limits_df = process_sign_info_file(sign_info_file)
    df = df.merge(limits_df, on='Tunnus')
    df["Speeders_count"] = (df["V1"] > df["Maks_kiirus"] + speeding_tolerance_kmh).astype("int8")

    hour_stats_df = (
        df.groupby(["Tunnus", "Year", "Month", "Day", "Hour", "Maks_kiirus"], observed=True) # add "X","Y" to groupby to see coordinate columns
          .agg(passers_count=("V1", "size"),
               avg_speed=("V1", "mean"),
               speeders_count=("Speeders_count", "sum"))
          .reset_index()
    )

    hour_stats_df["speeders_share"] = hour_stats_df["speeders_count"] / hour_stats_df["passers_count"]
    hour_stats_df = hour_stats_df[hour_stats_df["passers_count"] >= min_passing_cars]

    # lisa nädalapäev
    kuupaev = pd.to_datetime(
        hour_stats_df[["Year", "Month", "Day"]].rename(columns=str.lower)
    )
    hour_stats_df["weekday"] = kuupaev.dt.dayofweek  # 0 = esmaspäev

    return hour_stats_df

In [59]:
df = process_measured_speeds_file("data/vana/2_ulikooli.csv")
df.head(15)

,Tunnus,Year,Month,Day,Hour,Maks_kiirus,passers_count,avg_speed,speeders_count,speeders_share,weekday
0,2_ulikooli,2023,11,16,15,20,40,25.275000,19,0.475000,3
1,2_ulikooli,2023,11,16,16,20,184,21.820652,37,0.201087,3
2,2_ulikooli,2023,11,16,17,20,91,25.494505,39,0.428571,3
3,2_ulikooli,2023,11,16,18,20,121,22.768595,28,0.231405,3
4,2_ulikooli,2023,11,16,19,20,169,21.721893,24,0.142012,3
5,2_ulikooli,2023,11,16,20,20,162,22.716049,36,0.222222,3
6,2_ulikooli,2023,11,16,21,20,123,23.373983,38,0.308943,3
7,2_ulikooli,2023,11,16,22,20,154,24.272728,48,0.311688,3
8,2_ulikooli,2023,11,16,23,20,101,23.722773,25,0.247525,3
9,2_ulikooli,2023,11,17,0,20,17,31.000000,13,0.764706,4
